In [1]:
"""
# DriftWatch Synthetic ML Baseline Evaluation
## Purpose

Evaluate anomaly detection methods on synthetic telemetry data with known anomaly labels.

Because synthetic data provides ground-truth labels, this experiment compares detectors using:
- Precision
- Recall
- F1-score

Models evaluated:
- Isolation Forest
- One-Class SVM
- Local Outlier Factor
"""

'\n# DriftWatch Synthetic ML Baseline Evaluation\n## Purpose\n\nEvaluate anomaly detection methods on synthetic telemetry data with known anomaly labels.\n\nBecause synthetic data provides ground-truth labels, this experiment compares detectors using:\n- Precision\n- Recall\n- F1-score\n\nModels evaluated:\n- Isolation Forest\n- One-Class SVM\n- Local Outlier Factor\n'

In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

from config import SYNTHETIC_DB_PATH

In [3]:
def load_table(db_path, table):
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(f"SELECT * FROM {table}", conn)

features = load_table(SYNTHETIC_DB_PATH, "features")
label_metrics = load_table(SYNTHETIC_DB_PATH, "label_metrics")

df = features.merge(label_metrics, on="metrics_id", how="left")
df["label"] = df["label_id"].notna().astype(int)

df["label"].value_counts()

label
0    1211
1     243
Name: count, dtype: int64

In [4]:
feature_cols = [c for c in features.columns if c != "metrics_id"]

X = df[feature_cols].copy()
y = df["label"]

valid = X.notna().all(axis=1)

X = X[valid]
y = y[valid]

X_normal = X[y == 0]

In [5]:
scaler = StandardScaler()

scaler.fit(X_normal)

X_scaled = scaler.transform(X)
X_normal_scaled = scaler.transform(X_normal)

In [6]:
contamination = y.mean()

models = {
    "Isolation Forest": IsolationForest(
        contamination=contamination,
        random_state=42
    ),
    "One-Class SVM": OneClassSVM(
        kernel="rbf",
        nu=0.05
    ),
    "LOF": LocalOutlierFactor(
        n_neighbors=20,
        novelty=True
    )
}

predictions = {}

for name, model in models.items():
    model.fit(X_normal_scaled)
    predictions[name] = (model.predict(X_scaled) == -1).astype(int)

In [7]:
results = []

for name, pred in predictions.items():
    results.append({
        "model": name,
        "precision": precision_score(y, pred),
        "recall": recall_score(y, pred),
        "f1": f1_score(y, pred)
    })

pd.DataFrame(results).sort_values("f1", ascending=False)

,model,precision,recall,f1
2,LOF,0.469484,0.411523,0.438596
1,One-Class SVM,0.446281,0.222222,0.296703
0,Isolation Forest,0.174797,0.176955,0.175869
